# 01f — Extract HydroATLAS Spatial Context
**Data source:** [HydroSHEDS BasinATLAS](https://www.hydrosheds.org/page/basinatlas)

**Input:** `train_base.parquet`, `val_base.parquet` from notebook 00 output

**Output:** `hydroatlas.parquet` (one row per unique station)

**Estimated time:** ~5-10 min (downloading and spatial join)

> Enable Internet in Kaggle settings.

In [ ]:
# Install required packages if running in Kaggle environment
!pip install -q geopandas pyarrow requests tqdm fiona pyogrio

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import os, time, requests, logging
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# === Logging Setup ===
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-5s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('01f_hydroatlas')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titleweight': 'bold', 'font.size': 11})

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Column config
LAT_COL     = 'Latitude'
LON_COL     = 'Longitude'
STATION_COL = 'station_id'

# Helper function to find input file path in Kaggle
def find_file(filename, default_dir='/kaggle/working'):
    target = os.path.join(default_dir, filename)
    if os.path.exists(target):
        return target
    input_dir = '/kaggle/input'
    if os.path.exists(input_dir):
        for root, _, files in os.walk(input_dir):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(f"File {filename} not found in input/working directories.")

In [ ]:
# Load base data to get unique stations
base_train_path = find_file('train_base.parquet')
base_val_path   = find_file('val_base.parquet')

train_base = pd.read_parquet(base_train_path)
val_base   = pd.read_parquet(base_val_path)
all_data   = pd.concat([train_base, val_base], ignore_index=True)

unique_stations = all_data.groupby(STATION_COL)[[LAT_COL, LON_COL]].first().reset_index()
log.info(f'Loaded unique stations: {len(unique_stations)}')

---
## Download BasinATLAS GDB from Figshare

In [ ]:
url = 'https://figshare.com/ndownloader/files/20082137'
local_path = '/tmp/BasinATLAS_v1.gdb.zip'

if os.path.exists(local_path):
    log.info(f'File {local_path} already exists. Skipping download.')
else:
    log.info(f'Downloading BasinATLAS GDB zip to {local_path}...')
    response = requests.get(url, stream=True, timeout=900)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024 * 1024  # 1MB chunk
    
    progress_bar = tqdm(total=total_size, unit='iB', unit_scale=True, desc='BasinATLAS')
    with open(local_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=block_size):
            if chunk:
                f.write(chunk)
                progress_bar.update(len(chunk))
    progress_bar.close()
    log.info('BasinATLAS GDB downloaded successfully!')

---
## Load and Spatial Join

In [ ]:
log.info('Loading BasinATLAS GDB into GeoPandas...')
# Read GDB zip (loads first layer by default)
basin_gdf = gpd.read_file(f'zip://{local_path}')
log.info(f'Loaded {len(basin_gdf)} basins. Bounding box CRS: {basin_gdf.crs}')

# Create GeoDataFrame for unique stations
station_points = gpd.GeoDataFrame(
    unique_stations,
    geometry=gpd.points_from_xy(unique_stations[LON_COL], unique_stations[LAT_COL]),
    crs='EPSG:4326'
)

# Match CRS
if station_points.crs != basin_gdf.crs:
    station_points = station_points.to_crs(basin_gdf.crs)

log.info('Performing spatial join intersects...')
joined = gpd.sjoin(station_points, basin_gdf, how='left', predicate='intersects')
log.info(f'Spatial join done: {joined.shape}')

In [ ]:
# Filter and map columns of interest
# UP_AREA -> basin_upstream_area_km2
# POP -> basin_population
# AG -> basin_agriculture_pct
# SLOPE -> basin_slope_deg

hydro_df = pd.DataFrame({
    STATION_COL: joined[STATION_COL],
    LAT_COL: joined[LAT_COL],
    LON_COL: joined[LON_COL],
    'basin_upstream_area_km2': joined['UP_AREA'].fillna(0.0),
    'basin_population': joined['POP'].fillna(0.0),
    'basin_agriculture_pct': joined['AG'].fillna(0.0),
    'basin_slope_deg': joined['SLOPE'].fillna(0.0)
})

display(hydro_df.describe())

---
## Clean up and Save

In [ ]:
# Clean up large temp file to release Kaggle disk space
if os.path.exists(local_path):
    os.remove(local_path)
    log.info(f'Removed local zip file {local_path} to free disk space.')

out_path = f'{OUTPUT_DIR}/hydroatlas.parquet'
hydro_df.to_parquet(out_path, index=False)
size_kb = os.path.getsize(out_path) / 1024
log.info(f'Saved: {out_path} ({size_kb:.1f} KB, {len(hydro_df)} rows)')
print('\n=== DONE ===')
print('Output: hydroatlas.parquet')
print('Next: add this notebook output as dataset input for 01e')